# GDF Viewer for VS Code

Open this notebook in VS Code, select the `cosmos` Python kernel, and run the cells. Choose a recording from the dropdown, then rerun the cells below the picker to inspect it. Files are opened read-only with `preload=False`.

In [ ]:
from pathlib import Path
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
import mne
import pandas as pd
from IPython.display import display

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (
        root
        for root in search_roots
        if (root / "notebooks").is_dir() and (root / "data").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the bci_cleaning project")

data_candidates = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "raw" / "BCI Database",
    PROJECT_ROOT / "data" / "raw",
]
DATA_ROOT = next(
    (root for root in data_candidates if root.is_dir() and any(root.rglob("*.gdf"))),
    None,
)
if DATA_ROOT is None:
    raise FileNotFoundError("No GDF recordings were found in data/processed or data/raw")

gdf_files = sorted(DATA_ROOT.rglob("*.gdf"))
print(f"Found {len(gdf_files)} GDF recordings in {DATA_ROOT.relative_to(PROJECT_ROOT)}")

In [ ]:
gdf_picker = widgets.Dropdown(
    options=[(path.relative_to(DATA_ROOT).as_posix(), str(path)) for path in gdf_files],
    description="Recording:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "initial"},
)
display(gdf_picker)

After changing the recording above, run this cell and the remaining cells again.

In [ ]:
selected_gdf = Path(gdf_picker.value)
raw = mne.io.read_raw_gdf(selected_gdf, preload=False, verbose="ERROR")

summary = pd.Series(
    {
        "file": selected_gdf.relative_to(DATA_ROOT).as_posix(),
        "channels": raw.info["nchan"],
        "sampling_frequency_hz": raw.info["sfreq"],
        "samples": raw.n_times,
        "duration_seconds": raw.times[-1],
        "annotations": len(raw.annotations),
        "preloaded": raw.preload,
    },
    name="value",
)
summary.to_frame()

In [ ]:
channel_table = pd.DataFrame(
    {
        "channel": raw.ch_names,
        "type": raw.get_channel_types(),
    }
)
channel_table

In [ ]:
annotation_table = pd.DataFrame(
    {
        "onset_seconds": raw.annotations.onset,
        "duration_seconds": raw.annotations.duration,
        "description": raw.annotations.description,
    }
)
annotation_table


In [ ]:
%matplotlib inline

plot_start_seconds = 0
plot_duration_seconds = min(30, raw.times[-1])
raw.plot(
    start=plot_start_seconds,
    duration=plot_duration_seconds,
    n_channels=min(20, raw.info["nchan"]),
    scalings="auto",
    show=False,
    block=False,
    title=selected_gdf.name,
)
plt.show()

In [ ]:
performances_path = PROJECT_ROOT / "data" / "processed" / "Perfomances.csv"
performances = pd.read_csv(
    performances_path,
    sep=";",
    encoding="utf-8",
    skiprows=2,
)

performance_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
for column in performance_columns:
    performances[column] = pd.to_numeric(
        performances[column].astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    ).astype(float)

performances = performances.dropna(subset=["SUJ_gender", "EXP_gender", "Birth_year"])
performances = performances.drop(63)
performances = performances.drop(88)
performances["SUJ_gender"] = pd.to_numeric(performances["SUJ_gender"])
performances

In [ ]:
gender_counts = (
    performances["SUJ_gender"]
    .value_counts()
    .reindex([1, 2], fill_value=0)
    .astype(int)
)

ax = gender_counts.plot.bar(
    color=["steelblue", "coral"],
    figsize=(7, 5),
    rot=0,
)
ax.set_title("Participant Gender Distribution")
ax.set_xlabel("SUJ_gender")
ax.set_ylabel("Number of Participants")
ax.set_xticklabels(["1", "2"])
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

In [ ]:
birth = performances["Birth_year"].astype(int)

plt.figure(figsize=(8, 5))
plt.hist(birth, bins=25, color="skyblue", edgecolor="black")
plt.title("Distribution of Participant Birth Years")
plt.xlabel("Birth Year")
plt.ylabel("Number of Participants")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

sns.kdeplot(birth, fill=True, color="blue", alpha=0.5)
plt.title("KDE Plot with Seaborn")
plt.xlabel("Values")
plt.ylabel("Density")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (10,6))
axes[0,0].hist(performances["Perf_RUN_3"], bins = 25,range=(0, 101),color = "skyblue", edgecolor = "black")
axes[0,0].set_xlabel("Percentage of Successful Runs")
axes[0,1].hist(performances["Perf_RUN_4"], bins = 25,range=(0, 101), color = "skyblue", edgecolor = "black")

axes[1,0].hist(performances["Perf_RUN_5"], bins = 25,range=(0, 101), color = "skyblue", edgecolor = "black")

axes[1,1].hist(performances["Perf_RUN_6"], bins = 25,range=(0, 101), color = "skyblue", edgecolor = "black")
plt.tight_layout()
plt.show()
'''
df = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022],
    'Sales_A': [120, 135, 150, 160],
    'Sales_B': [80, 95, 100, 120],
    'Profit_A': [30, 32, 36, 38],
    'Profit_B': [18, 20, 22, 26]
})

fig, axs = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)

# [0, 0] Sales of A
axs[0, 0].bar(df['Year'], df['Sales_A'], color='steelblue')
axs[0, 0].set_title("Sales - Product A")

# [0, 1] Sales of B
axs[0, 1].bar(df['Year'], df['Sales_B'], color='salmon')
axs[0, 1].set_title("Sales - Product B")

# [1, 0] Profit of A
axs[1, 0].plot(df['Year'], df['Profit_A'], color='seagreen', marker='o')
axs[1, 0].set_title("Profit - Product A")

# [1, 1] Profit of B
axs[1, 1].plot(df['Year'], df['Profit_B'], color='orange', marker='o')
axs[1, 1].set_title("Profit - Product B")


for ax in axs[1, :]:
    ax.set_xticks(df['Year'])
    ax.set_xticklabels(df['Year'])

for ax in axs[:, 0]:   
    ax.set_ylabel("Amount (k$)")

plt.tight_layout()
plt.show()
'''